# 3. Graphiques interactifs avec Plotly

Plotly.js est utilisé par défaut. On génère du HTML contenant le graphique et on l'affiche avec `Deno.jupyter.display`.

## 3.0 Fonction d'affichage

La fonction `plotly(data, layout)` ci-dessous affiche un graphique [Plotly.js](https://plotly.com/javascript/). Exécute cette cellule une fois, puis réutilise la fonction dans les cellules suivantes.

In [ ]:
function plotly(data: object[], layout: object = {}) {
  const id = `plot-${crypto.randomUUID()}`;
  const html = `
<div id="${id}" style="width: 100%; max-width: 750px; height: 450px;"></div>
<script>
  (function () {
    const draw = () => Plotly.newPlot("${id}", ${JSON.stringify(data)}, ${JSON.stringify(layout)}, { responsive: true });
    if (window.Plotly) return draw();
    const script = document.createElement("script");
    script.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    script.onload = draw;
    document.head.appendChild(script);
  })();
</script>`;
  Deno.jupyter.display({ "text/html": html }, { raw: true });
}

## 3.1 Graphique en barres

In [ ]:
plotly(
  [{ x: ["Littéraire", "Scientifique", "Économique", "Artistique"], y: [45, 80, 60, 30], type: "bar", marker: { color: ["#ff6384", "#36a2eb", "#ffce56", "#4bc0c0"] } }],
  { title: "Répartition des élèves par filière", xaxis: { title: "Filières" }, yaxis: { title: "Nombre d'élèves" } },
);

## 3.2 Graphique en secteurs (camembert)

In [ ]:
plotly(
  [{ labels: ["Apple", "Samsung", "Xiaomi", "Autres"], values: [40, 35, 15, 10], type: "pie" }],
  { title: "Parts de marché des fabricants de téléphones", height: 450 },
);

## 3.3 Nuage de points

In [ ]:
plotly(
  [{ x: [1, 2, 3, 4, 5, 6, 7, 8], y: [40, 50, 55, 60, 70, 75, 80, 85], mode: "markers", type: "scatter", marker: { size: 12, color: "#36a2eb" } }],
  { title: "Corrélation entre heures d'étude et notes", xaxis: { title: "Heures d'étude" }, yaxis: { title: "Note (%)" } },
);

## 3.4 Graphique de séries temporelles

In [ ]:
plotly(
  [{ x: ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"], y: [12, 14, 15, 13, 11, 9, 10], mode: "lines+markers", type: "scatter", line: { color: "#4bc0c0" } }],
  { title: "Évolution des températures sur une semaine", xaxis: { title: "Jour" }, yaxis: { title: "Température (°C)" } },
);

## 3.5 Histogramme

In [ ]:
plotly(
  [{ x: [15, 17, 18, 19, 21, 21, 22, 23, 24, 24, 25, 25, 26, 27, 30], type: "histogram", nbinsx: 6, marker: { color: "#9966ff" } }],
  { title: "Distribution des âges des participants", xaxis: { title: "Âge" }, yaxis: { title: "Fréquence" } },
);

## 3.6 Graphique avec des données réelles (Polars + Plotly)

In [ ]:
import pl from "nodejs-polars";

const df = pl.readCSV("../data/titanic.csv");

const survivalByClass = df
  .groupBy("Pclass")
  .agg(pl.col("Survived").mean().alias("survival_rate"))
  .sort("Pclass");

const rows = survivalByClass.toRecords();
const classes = rows.map((r: { Pclass: number }) => `Classe ${r.Pclass}`);
const rates = rows.map((r: { survival_rate: number }) => Number(r.survival_rate));

const data = [{ x: classes, y: rates, type: "bar", marker: { color: ["#ff6384", "#36a2eb", "#ffce56"] } }];
const layout = { title: "Taux de survie par classe (Titanic)", xaxis: { title: "Classe" }, yaxis: { title: "Taux de survie", range: [0, 1] } };

plotly(data, layout);

## 3.7 Sauvegarder un graphique en image

In [ ]:
// Dans un notebook, le graphique est interactif.
// Pour exporter une image, utiliser l'icône de caméra dans la barre d'outils Plotly,
// ou générer le graphique avec Plotly.toImage côté client.